# 06 — Fenotipi con geometria di RIEMANN (SPD manifold)
Le matrici di covarianza/connettività sono **SPD** (vivono su una varietà, non in spazio euclideo).
Clusterizzarle col tangent space di Riemann è la scelta principled (portato da **V5W_07**).

**Vantaggio chiave**: la metrica affine-invariante **ignora la scala globale** → possiamo dimostrare
che lo split è per **struttura di connettività**, non per ampiezza/SNR (controllo det=1). L'euclideo
non può farlo.

Pipeline: covarianze per-trial (OAS) → media di Riemann per soggetto → tangent space → cluster k=2,
con permutation test e controlli anti-artefatto. Poi decoding Riemann (MDM, TS+LR) e fenotipo→decodabilità.

> ⚠️ 15 soggetti = esplorativo (la tesi ne aveva 74).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import numpy as np, matplotlib.pyplot as plt
import track3_config as C, track3_io as io, track3_train as T
import track3_phenotypes as PH
from sklearn.metrics import adjusted_rand_score
from scipy.stats import mannwhitneyu
print(C.summary()); assert C.DATA_ROOT is not None, C._no_data_msg()

## §1 — Covarianze SPD + media di Riemann per soggetto

In [ ]:
data = PH.load_covariances()   # OAS covariances, broadband; ~1-2 min
print('covarianze per-trial:', data['covs'].shape, '| medie-soggetto SPD:', data['M'].shape)
clab = data['clab']

## §2 — Clustering nel tangent space + permutation test
La silhouette k=2 va confrontata con la distribuzione nulla (permutazione delle feature): se osservata >> nulla, la separazione è genuina (non circular analysis).

In [ ]:
labels, TS, Z = PH.riemann_cluster(data['M'], k=2)
print('cluster sizes:', np.bincount(labels))
obs, null, p = PH.silhouette_permutation(TS, k=2, n_perm=2000)
print(f'silhouette k=2 (tangent) = {obs:.3f}  null={null.mean():.3f}±{null.std():.3f}  p={p:.4f}  '
      + ('GENUINO' if p<0.05 else 'n.s.'))
fig, ax = plt.subplots(1,2, figsize=(12,4.5))
for c in (0,1):
    m=labels==c; ax[0].scatter(Z[m,0], Z[m,1], s=80, label=f'C{c} (n={m.sum()})', edgecolors='white')
for i,s in enumerate(data['ids']): ax[0].annotate(f'S{s:02d}',(Z[i,0],Z[i,1]),fontsize=7,xytext=(3,3),textcoords='offset points')
ax[0].set_xlabel('Tangent PC1'); ax[0].set_ylabel('Tangent PC2'); ax[0].set_title('Fenotipi (Riemann tangent)'); ax[0].legend()
ax[1].hist(null, bins=30, color='0.7', edgecolor='white'); ax[1].axvline(obs, color='crimson', lw=2.5, label=f'oss={obs:.3f}')
ax[1].set_title(f'Permutation silhouette k=2  p={p:.3f}'); ax[1].set_xlabel('silhouette'); ax[1].legend()
plt.tight_layout(); plt.savefig(C.FIG_DIR/'pheno_riemann_cluster.png', dpi=130); plt.show()

## §3 — Controllo anti-artefatto (1): scala vs struttura
Frobenius/traccia (= ampiezza globale) **possono** differire, ma la metrica affine-invariante li ignora.
**Test det=1**: normalizzo ogni media-soggetto a determinante 1 (rimuovo la scala) e **ri-clusterizzo**.
Se l'ARI col clustering originale è alto → lo split è per **struttura**, non ampiezza.

In [ ]:
fro = np.array([np.linalg.norm(data['M'][i],'fro') for i in range(len(data['ids']))])
tra = np.array([np.trace(data['M'][i]) for i in range(len(data['ids']))])
for name,arr in [('Frobenius',fro),('trace',tra)]:
    _,pv = mannwhitneyu(arr[labels==0], arr[labels==1])
    print(f'{name}: C0={arr[labels==0].mean():.2e} C1={arr[labels==1].mean():.2e} p(MW)={pv:.3f}')
Mn = PH.det1_normalize(data['M'])
labels_n, TSn, _ = PH.riemann_cluster(Mn, k=2)
ari_det1 = adjusted_rand_score(labels_n, labels)
print(f'\nARI(det=1 vs originale) = {ari_det1:.3f}  '
      + ('=> split PER STRUTTURA (sopravvive alla rimozione di scala)' if ari_det1>0.5 else '=> split guidato da scala/ampiezza ⚠️'))

## §4 — Controllo anti-artefatto (2): covarianze pulite (1–30 Hz, no frontopolari)
Ri-clusterizzo su covarianze filtrate 1–30 Hz e **senza canali frontopolari/EOG** (fonte di artefatti
oculari). Se l'ARI resta alto → lo split non è guidato da artefatti oculari/gamma.

In [ ]:
keep = PH.frontopolar_keep_idx(clab)
print(f'canali tenuti: {len(keep)}/{len(clab)} (rimossi i frontopolari/EOG)')
data_clean = PH.load_covariances(band=(1,30), keep_idx=keep)   # ~1-2 min
labels_c, TSc, _ = PH.riemann_cluster(data_clean['M'], k=2)
ari_clean = adjusted_rand_score(labels_c, labels)
print(f'ARI(clean 1-30Hz no-EOG vs originale) = {ari_clean:.3f}  '
      + ('=> robusto agli artefatti' if ari_clean>0.5 else '=> possibile contributo di artefatti ⚠️'))

## §5 — Decoding Riemann subject-specific (gold standard classico)
MDM e TangentSpace+LogReg: se anche questi decoder classici forti sono al chance, il ceiling è blindato;
se decodificano, sono un baseline competitivo con le ConvNet.

In [ ]:
rows = PH.riemann_decoding(data)
import pandas as pd
df_dec = pd.DataFrame(rows, columns=['subj','MDM','TS_LR']).set_index('subj')
df_dec.to_csv(C.RESULTS_DIR/'riemann_decoding.csv')
print(f"MDM   : mean={df_dec.MDM.mean():.3f}  >chance={(df_dec.MDM>C.CHANCE_LEVEL).mean()*100:.0f}%")
print(f"TS+LR : mean={df_dec.TS_LR.mean():.3f}  >chance={(df_dec.TS_LR>C.CHANCE_LEVEL).mean()*100:.0f}%")
print(f'(chance {C.CHANCE_LEVEL}) — confronto: Shallow dep 0.575, EEGNet dep 0.555)')
df_dec.round(3)

## §6 — Il fenotipo (Riemann) predice la decodabilità?
Domanda chiave: i soggetti di un fenotipo si decodificano meglio? (Mann-Whitney C0 vs C1).

In [ ]:
for mname in ['MDM','TS_LR']:
    dvals = df_dec[mname].values
    b0, b1 = dvals[labels==0], dvals[labels==1]
    _,pv = mannwhitneyu(b0,b1)
    print(f'{mname}: C0={b0.mean():.3f}  C1={b1.mean():.3f}  p(MW)={pv:.3f}')
fig, ax = plt.subplots(figsize=(6,4))
for c,(col) in zip((0,1),('#4DA3FF','#FF8C42')):
    v=df_dec['TS_LR'].values[labels==c]; ax.scatter(np.full(len(v),c)+ (np.arange(len(v))-len(v)/2)*0.02, v, color=col, s=55, edgecolor='w')
ax.set_xticks([0,1]); ax.set_xticklabels(['C0','C1']); ax.set_ylabel('bAcc (TS+LR)')
ax.axhline(C.CHANCE_LEVEL, color='r', ls='--', label='chance'); ax.set_title('Fenotipo Riemann vs decodabilità'); ax.legend()
plt.tight_layout(); plt.savefig(C.FIG_DIR/'pheno_riemann_vs_decod.png', dpi=130); plt.show()

## §7 — Firma spaziale: dove differiscono i fenotipi (node strength)
Topomap della differenza di forza di connettività C1 − C0 (dalle medie-soggetto SPD).

In [ ]:
import mne
clab_pos, pos = io.canonical_positions()
info = mne.create_info(clab_pos, C.FS, ch_types='eeg')
info.set_montage(mne.channels.make_dig_montage(ch_pos={c:p for c,p in zip(clab_pos,pos)}, coord_frame='head'), on_missing='warn')
ns0 = data['M'][labels==0].mean(0); ns1 = data['M'][labels==1].mean(0)
diff = ns1.mean(1) - ns0.mean(1)
fig, ax = plt.subplots(figsize=(4.5,4))
mne.viz.plot_topomap(diff, info, axes=ax, show=False, cmap='RdBu_r'); ax.set_title('Node strength C1 − C0 (Riemann)')
plt.tight_layout(); plt.savefig(C.FIG_DIR/'pheno_riemann_topomap.png', dpi=130); plt.show()
print('C1>C0:', [clab_pos[i] for i in np.argsort(diff)[::-1][:6]])
print('C0>C1:', [clab_pos[i] for i in np.argsort(diff)[:6]])

## Conclusioni
- **Fenotipi genuini?** silhouette tangent + permutation (§2). 
- **Struttura vs ampiezza** (§3): ARI det=1 alto → la separazione NON è SNR/scala (prova che l'euclideo non dà).
- **Robusto agli artefatti** (§4): ARI con covarianze pulite 1–30 Hz senza frontopolari.
- **Decoding Riemann** (§5): MDM/TS+LR come baseline classico forte.
- **Fenotipo → decodabilità** (§6): il ponte fenotipo-performance.

Se det=1 e clean confermano (ARI alto) e TS+LR decodifica, questo è materiale solido: i fenotipi di
connettività sono reali e strutturali anche su Track#3, e il metodo Riemann è più principled dell'euclideo (nb 05).